<a href="https://colab.research.google.com/github/fardinAhmed285/tsp-crossover-study/blob/sparsh%2Ftsp_model/tsp_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROBLEM 5: TRAVELLING SALESMAN PROBLEM (TSP)
## Comparative Study of Crossover Operators

---

### Dataset Information

- **Dataset:** ulysses22 (TSPLIB)
- **Number of Cities:** 22
- **Target Optimal Tour Distance:** 7013
- **Problem Type:** Travelling Salesman Problem (TSP)

### Experiment Objectives

The objective of this experiment is to compare three different Genetic Algorithm crossover operators:

1. **ER (Edge Recombination):** Preserves adjacent city connections from parent tours.
2. **PMX (Partially Mapped Crossover):** Preserves positional information from parent tours.
3. **OX (Order Crossover):** Preserves the relative ordering of cities.

### Experimental Configuration

| Parameter | Value |
|:---|:---|
| Population Size | 100 |
| Independent Runs | 30 per operator |
| Total Runs | 90 |
| Generations | 1000 |
| Crossover Probability | 0.8 |
| Mutation Probability | 0.2 |
| Elitism | 2 individuals |
| Selection Method | Tournament Selection (size 2) |
| Mutation Method | Inversion Mutation |
| Random Seed | 2026 onwards |

### Evaluation Metrics

- Final Best Tour Distance
- Convergence History
- Edge Heritability
- Mann-Whitney U Statistical Test
- Vargha-Delaney A12 Effect Size

### Expected Outcome

Identify the best-performing crossover operator based on tour distance, convergence behaviour, edge heritability, and statistical significance.

In [16]:
%pip install -q deap

In [17]:
import math
import random
import csv
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from deap import base, creator, tools

# STAGE 1: DATASET LOADING AND DISTANCE CALCULATION

---

### Objective

Load the TSPLIB ulysses22 dataset and calculate the geographical distances between all 22 cities.

### Dataset Description

The ulysses22 dataset contains geographical coordinates representing 22 cities.

The dataset is embedded directly into the notebook, making the implementation standalone without requiring manual file uploads.

### Distance Calculation

We use the **TSPLIB GEO distance calculation method** to calculate the distance between two cities.

The GEO distance formula uses:

- Latitude and longitude coordinates.
- Conversion from degrees.minutes format into radians.
- Earth's radius of 6378.388 km.
- Spherical distance calculation.

### Expected Output

- Parsed geographical coordinates.
- Number of cities: 22.
- Distance matrix of dimensions 22 × 22.

**Reference Optimal Tour Distance: 7013**

In [18]:
ULYSSES22_TSPLIB_DATA = """
NAME: ulysses22.tsp
TYPE: TSP
COMMENT: Odyssey of Homer (Groetschel/Padberg)
DIMENSION: 22
EDGE_WEIGHT_TYPE: GEO
NODE_COORD_SECTION
1 38.24 20.42
2 39.57 26.15
3 40.56 25.32
4 36.26 23.12
5 33.48 26.54
6 37.56 25.32
7 38.42 20.27
8 37.03 22.49
9 35.18 19.03
10 36.28 16.53
11 37.10 14.40
12 39.15 15.21
13 38.15 15.35
14 37.23 15.34
15 36.47 15.13
16 38.00 13.04
17 37.30 11.02
18 40.21 13.48
19 40.13 14.28
20 40.10 14.09
21 40.30 13.03
22 39.41 10.12
EOF
"""

In [19]:
def parse_ulysses22_coords(data_str):

    # Parses latitude and longitude coordinates for all 22 cities from the raw text string above.

    coords = []
    lines = data_str.strip().split('\n')
    in_section = False
    for line in lines:
        line = line.strip()
        if line == "NODE_COORD_SECTION":
            in_section = True
            continue
        if line == "EOF" or not line:
            break
        if in_section:
            parts = line.split()
            lat = float(parts[1])
            lon = float(parts[2])
            coords.append((lat, lon))
    return coords

In [20]:
def geo_distance(coord1, coord2):

    # Calculates exact TSPLIB GEO distance between two cities on Earth.
    # TSPLIB uses a specialized formula converting (degrees.minutes) into radians.
    # Using this exact formula ensures our optimal score target is 7013.

    PI = 3.14159265358979323846
    RR = 6378.388  # Earth's radius in kilometers

    lat1, lon1 = coord1
    lat2, lon2 = coord2

    # Convert latitude 1 from deg.min format into radians
    deg1 = int(lat1)
    min1 = lat1 - deg1
    rad_lat1 = PI * (deg1 + 5.0 * min1 / 3.0) / 180.0

    # Convert longitude 1
    deg1 = int(lon1)
    min1 = lon1 - deg1
    rad_lon1 = PI * (deg1 + 5.0 * min1 / 3.0) / 180.0

    # Convert latitude 2
    deg2 = int(lat2)
    min2 = lat2 - deg2
    rad_lat2 = PI * (deg2 + 5.0 * min2 / 3.0) / 180.0

    # Convert longitude 2
    deg2 = int(lon2)
    min2 = lon2 - deg2
    rad_lon2 = PI * (deg2 + 5.0 * min2 / 3.0) / 180.0

    # Calculate spherical distance
    q1 = math.cos(rad_lon1 - rad_lon2)
    q2 = math.cos(rad_lat1 - rad_lat2)
    q3 = math.cos(rad_lat1 + rad_lat2)
    d = int(RR * math.acos(0.5 * ((1.0 + q1) * q2 - (1.0 - q1) * q3)) + 1.0)
    return d

In [21]:
def compute_distance_matrix(coords):
    # Precomputes a 22 x 22 matrix of distance lookups so we don't have to re-calculate math formulas millions of times during evolution.
    n = len(coords)
    dist_matrix = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            if i != j:
                dist_matrix[i][j] = geo_distance(coords[i], coords[j])
    return dist_matrix

# Global data setup
COORDS = parse_ulysses22_coords(ULYSSES22_TSPLIB_DATA)
N_CITIES = len(COORDS)
DIST_MATRIX = compute_distance_matrix(COORDS)

# STAGE 2: FITNESS EVALUATION AND EDGE HERITABILITY

---

### Objective

Define the fitness function and edge heritability metric used to evaluate the quality of generated TSP tours.

### 1. Fitness Evaluation

The fitness function calculates the total distance of a complete tour.

It considers:

- Distance between consecutive cities.
- Distance from the last city back to the starting city.

**Goal:** Minimize the total tour distance.

### 2. Tour Edge Extraction

Each tour consists of 22 undirected edges connecting consecutive cities.

The edge extraction function identifies all connections in a tour.

### 3. Edge Heritability

Edge heritability measures the proportion of edges in a child tour that are inherited from either of its two parent tours.

\[
H(C\mid P_1,P_2)=
\frac{|E(C)\cap(E(P_1)\cup E(P_2))|}{|E(C)|}
\]

Where:

- \(E(C)\): Edges of the child.
- \(E(P_1)\): Edges of parent 1.
- \(E(P_2)\): Edges of parent 2.

### Interpretation

- **1.0:** All child edges are inherited from the parents.
- **0.5:** Half of the child edges are inherited.
- **0.0:** No child edges are inherited.

A higher value indicates greater preservation of parental edge information.

In [22]:
def eval_tsp(individual):
    # Fitness Function: Sums the distances between each city in the tour, including the return step from the last city back to the first.
    tour_len = 0
    n = len(individual)
    for i in range(n):
        u = individual[i]
        v = individual[(i + 1) % n]
        tour_len += DIST_MATRIX[u][v]
    return (tour_len,)

In [23]:
def get_tour_edges(individual):
    # Extracts all 22 unique undirected city connections (edges) from a tour.
    # For example: a tour [0, 1, 2] has edges (0,1), (1,2), and (0,2).

    n = len(individual)
    edges = set()
    for i in range(n):
        u = individual[i]
        v = individual[(i + 1) % n]
        edges.add(tuple(sorted((u, v))))
    return edges

In [24]:
def calculate_edge_heritability(child, p1, p2):
    # Edge Heritability Metric: Measures what fraction (0.0 to 1.0) of the child's 22 edges actually came from its two parents combined.

    c_edges = get_tour_edges(child)
    p_edges = get_tour_edges(p1).union(get_tour_edges(p2))
    return len(c_edges.intersection(p_edges)) / float(len(child))

# STAGE 3: CROSSOVER OPERATORS AND MUTATION

---

### Objective

Implement three different crossover operators and one mutation operator for the Genetic Algorithm.

### 1. Edge Recombination (ER)

ER focuses on preserving the adjacency relationships between cities.

**Working Principle:**

- Construct an edge map using both parent tours.
- Select a starting city.
- Identify unvisited neighbouring cities.
- Prefer the neighbour with the fewest remaining connections.
- Continue until all cities are included.

**Main Advantage:** Attempts to preserve parent edges.

### 2. Partially Mapped Crossover (PMX)

PMX preserves positional information using a mapping relationship between two parent tours.

**Main Advantage:** Maintains valid permutations while transferring positional information.

### 3. Order Crossover (OX)

OX preserves the relative ordering of cities from parent tours.

**Main Advantage:** Transfers useful ordering information while maintaining valid tours.

### 4. Inversion Mutation

Inversion mutation selects two random positions and reverses the sequence of cities between them.

**Main Advantage:** Introduces diversity while retaining most existing tour adjacencies.

### Comparison Focus

The three crossover operators will be compared based on:

- Tour quality.
- Preservation of parental edges.
- Convergence behaviour.

In [25]:
def cx_edge_recombination(ind1, ind2):
    # Edge Recombination Crossover (ER)

    n = len(ind1)

    def generate_er_child(parent_a, parent_b):
        # 1. Build combined edge map showing all neighbors of each city in both parents
        edge_map = {i: set() for i in range(n)}
        for p in (parent_a, parent_b):
            for i in range(n):
                u = p[i]
                v = p[(i + 1) % n]
                edge_map[u].add(v)
                edge_map[v].add(u)

        # 2. Pick a random starting city
        curr = random.choice([parent_a[0], parent_b[0]])
        child = [curr]
        visited = {curr}

        # 3. Step through remaining cities
        while len(child) < n:
            # Remove current city from all neighbor lists
            for neighbors in edge_map.values():
                neighbors.discard(curr)

            unvisited_neighbors = [node for node in edge_map[curr] if node not in visited]

            if unvisited_neighbors:
                # Pick neighbor with fewest remaining connections
                min_deg = min(len(edge_map[node]) for node in unvisited_neighbors)
                candidates = [node for node in unvisited_neighbors if len(edge_map[node]) == min_deg]
                nxt = random.choice(candidates)
            else:
                # If dead-end, pick a random unvisited city
                unvisited = [node for node in range(n) if node not in visited]
                nxt = random.choice(unvisited)

            curr = nxt
            child.append(curr)
            visited.add(curr)

        return child

    # Generate two children in-place to fit DEAP's expected framework
    c1 = generate_er_child(ind1, ind2)
    c2 = generate_er_child(ind2, ind1)
    ind1[:], ind2[:] = c1, c2
    return ind1, ind2

In [26]:
def mut_inversion(individual):
    # Inversion Mutation

    n = len(individual)
    i, j = sorted(random.sample(range(n), 2))
    individual[i:j] = list(reversed(individual[i:j]))
    return (individual,)

# STAGE 4: DEAP FRAMEWORK INITIALIZATION

---

### Objective

Initialize the DEAP evolutionary computation framework and configure the Genetic Algorithm components.

### Components

**1. Fitness Function**

- Uses the TSP tour distance.
- Minimization objective.
- Lower fitness values represent better solutions.

**2. Individual Representation**

- Each individual represents a complete TSP tour.
- A tour is represented as a permutation of 22 cities.

**3. Population Initialization**

- Generates random permutations of the city indices.
- Creates an initial population of 100 individuals.

**4. Selection Method**

- Tournament Selection.
- Tournament Size: 2.

**5. Mutation Operator**

- Inversion Mutation.

**6. Crossover Operator**

The toolbox dynamically registers one of the following:

- ER: Edge Recombination.
- PMX: Partially Mapped Crossover.
- OX: Order Crossover.

### Important

All three crossover operators use the same population size, selection method, mutation method, and evolutionary parameters to ensure a controlled comparison.

In [27]:
if not hasattr(creator, "FitnessMin"):
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMin)

def build_toolbox(cx_operator_name):
    # Sets up DEAP operators for the specified crossover algorithm (ER, PMX, or OX).

    toolbox = base.Toolbox()
    toolbox.register("indices", random.sample, range(N_CITIES), N_CITIES)
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", eval_tsp)
    toolbox.register("select", tools.selTournament, tournsize=2)
    toolbox.register("mutate", mut_inversion)

    if cx_operator_name == "ER":
        toolbox.register("mate", cx_edge_recombination)
    elif cx_operator_name == "PMX":
        toolbox.register("mate", tools.cxPartialyMatched)
    elif cx_operator_name == "OX":
        toolbox.register("mate", tools.cxOrdered)
    else:
        raise ValueError(f"Unknown crossover operator: {cx_operator_name}")

    return toolbox

# STAGE 5: SINGLE EVOLUTIONARY ALGORITHM RUN

---

### Objective

Implement the complete Genetic Algorithm for one independent experimental run.

Each run executes for 1000 generations using the selected crossover operator.

### Evolutionary Process

**Step 1: Population Initialization**

Generate a population of 100 random TSP tours.

**Step 2: Fitness Evaluation**

Calculate the tour distance of every individual.

**Step 3: Elitism**

Preserve the two best individuals unchanged for the next generation.

**Step 4: Parent Selection**

Select individuals using Tournament Selection with tournament size 2.

**Step 5: Crossover**

Apply the selected crossover operator with probability 0.8.

**Step 6: Edge Heritability Calculation**

Measure the proportion of child edges inherited from the two parents.

**Step 7: Mutation**

Apply inversion mutation with probability 0.2.

**Step 8: Fitness Re-evaluation**

Recalculate the fitness of modified offspring.

**Step 9: Population Replacement**

Combine the elite individuals with the offspring.

**Step 10: Performance Recording**

Record:

- Best tour distance.
- Mean edge heritability.

### Output

The function returns:

1. Final best tour distance.
2. Best distance history across generations.
3. Edge heritability history across generations.

In [28]:
def run_single_ea(cx_name, seed, pop_size=100, n_gens=1000, cx_pb=0.8, mut_pb=0.2, n_elites=2):
    # Executes a single 1000-generation GA trial for a specific operator and seed.

    # Fix random seed for exact scientific reproducibility
    random.seed(seed)
    np.random.seed(seed)

    toolbox = build_toolbox(cx_name)
    pop = toolbox.population(n=pop_size)

    # Evaluate initial random population
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)

    best_distance_history = []
    heritability_history = []

    # Main Evolutionary Loop
    for gen in range(n_gens):
        # Step A: Elitism (Keep top 2 best tours unchanged)
        elites = [toolbox.clone(ind) for ind in tools.selBest(pop, n_elites)]

        # Step B: Tournament Selection for the remaining slots
        offspring = toolbox.select(pop, pop_size - n_elites)
        offspring = [toolbox.clone(ind) for ind in offspring]

        # Step C: Apply Crossover & Measure Heritability
        gen_heritability_vals = []
        for i in range(0, len(offspring) - 1, 2):
            if random.random() < cx_pb:
                p1 = toolbox.clone(offspring[i])
                p2 = toolbox.clone(offspring[i + 1])

                toolbox.mate(offspring[i], offspring[i + 1])

                # Calculate how many parent edges survived in offspring
                h1 = calculate_edge_heritability(offspring[i], p1, p2)
                h2 = calculate_edge_heritability(offspring[i + 1], p1, p2)
                gen_heritability_vals.extend([h1, h2])

                del offspring[i].fitness.values
                del offspring[i + 1].fitness.values

        # Step D: Apply Inversion Mutation
        for ind in offspring:
            if random.random() < mut_pb:
                toolbox.mutate(ind)
                del ind.fitness.values

        # Step E: Re-evaluate changed offspring
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid_ind:
            ind.fitness.values = toolbox.evaluate(ind)

        # Combine elites + offspring to form next generation population
        pop = elites + offspring

        # Step F: Record generation metrics
        best_ind = tools.selBest(pop, 1)[0]
        best_distance_history.append(best_ind.fitness.values[0])

        mean_h = np.mean(gen_heritability_vals) if gen_heritability_vals else 0.0
        heritability_history.append(mean_h)

    final_best_dist = best_distance_history[-1]
    return final_best_dist, best_distance_history, heritability_history

# STAGE 6: STATISTICAL TEST HELPERS

---

### Objective

Define statistical methods to compare the performance of different crossover operators.

### 1. Mann-Whitney U Test

The Mann-Whitney U test is a non-parametric statistical test used to compare two independent samples.

It determines whether there is evidence of a difference between the distributions of two crossover operators.

**Hypotheses:**

- **H₀:** There is no difference between the two operator distributions.
- **H₁:** There is a difference between the two operator distributions.

**Significance Level:**

\[
\alpha=0.05
\]

**Interpretation:**

- \(p<0.05\): Statistically significant difference.
- \(p\geq0.05\): Insufficient evidence of a statistically significant difference.

### 2. Vargha-Delaney A12 Effect Size

A12 measures the probability that a randomly selected result from one sample is better than a randomly selected result from another sample, with ties counted as half.

For tour minimization:

- **A12 = 0.5:** Similar performance.
- **A12 < 0.5:** Sample 1 tends to achieve lower tour distances.
- **A12 > 0.5:** Sample 2 tends to achieve lower tour distances.

### Comparisons

- ER vs PMX
- ER vs OX

Both statistical significance and effect size are considered when interpreting the results.

In [29]:
def vargha_delaney_a12(sample1, sample2):
    # Computes Vargha-Delaney A12 effect size statistic.

    m = len(sample1)
    n = len(sample2)
    r = stats.rankdata(np.concatenate([sample1, sample2]))
    r1 = np.sum(r[:m])
    a = (r1 / m - (m + 1) / 2.0) / n
    return a

# STAGE 7: MAIN EXPERIMENT EXECUTION

---

### Objective

Execute 90 independent Genetic Algorithm runs to compare the three crossover operators.

### Experimental Configuration

| Parameter | Value |
|:---|:---|
| Crossover Operators | ER, PMX, OX |
| Runs per Operator | 30 |
| Total Runs | 90 |
| Generations per Run | 1000 |
| Population Size | 100 |
| Base Random Seed | 2026 |

### Execution Procedure

1. Execute 30 independent runs using ER.
2. Execute 30 independent runs using PMX.
3. Execute 30 independent runs using OX.
4. Store the final best tour distance from every run.
5. Store the convergence history.
6. Store the edge heritability history.
7. Calculate mean convergence and mean edge heritability across the 30 runs.

### Statistical Summary

The following descriptive statistics are calculated for each operator:

- Median.
- Mean.
- Standard Deviation.
- Minimum.
- Maximum.

### Statistical Comparison

The Mann-Whitney U test and Vargha-Delaney A12 effect size are calculated for:

- ER vs PMX.
- ER vs OX.

### Result Export

The statistical summary is exported to:

`tsp_results_summary.csv`

# STAGE 8: VISUALIZATION AND PERFORMANCE ANALYSIS

---

### Objective

Visualize and compare the performance of ER, PMX, and OX crossover operators.

Three plots are generated using the experimental results.

### FIGURE 1: TOUR DISTANCE CONVERGENCE

**Purpose:** Compare the convergence behaviour of the three crossover operators over 1000 generations.

- X-axis: Generation.
- Y-axis: Mean Best Tour Distance.
- Blue: ER.
- Orange: PMX.
- Green: OX.
- Dashed horizontal line: TSPLIB optimal distance (7013).

**Interpretation:** A faster decrease in tour distance indicates faster convergence towards better solutions.

### FIGURE 2: EDGE HERITABILITY

**Purpose:** Compare the preservation of parental edges across generations.

- X-axis: Generation.
- Y-axis: Mean Edge Heritability.
- Blue: ER.
- Orange: PMX.
- Green: OX.

**Interpretation:** Higher edge heritability indicates greater preservation of parental connections during crossover.

### FIGURE 3: FINAL TOUR DISTANCE BOXPLOT

**Purpose:** Compare the distribution of final best tour distances across 30 independent runs.

The boxplot displays:

- Median.
- Interquartile range.
- Distribution of results.
- Potential outliers.

The optimal TSPLIB distance of 7013 is included as a reference line.

### Generated Visualization Files

1. `tsp_convergence.png`
2. `tsp_heritability.png`
3. `tsp_boxplot.png`

These figures can be included in the final experimental report.

In [30]:
def main():
    print("=" * 75)
    print("  TSP CROSSOVER OPERATOR COMPARISON: ulysses22 (TSPLIB)")
    print("  Target Optimal Distance: 7013")
    print("=" * 75)

    OPERATORS = ["ER", "PMX", "OX"]
    N_RUNS = 30
    N_GENS = 1000
    POP_SIZE = 100
    BASE_SEED = 2026

    results_data = {op: [] for op in OPERATORS}
    convergence_histories = {op: [] for op in OPERATORS}
    heritability_histories = {op: [] for op in OPERATORS}

    # Run 30 Seeds per Operator (90 Total Evolutionary Runs)
    for op in OPERATORS:
        print(f"\n[+] Executing {N_RUNS} independent runs for operator: {op} ...")
        for r in range(N_RUNS):
            seed = BASE_SEED + r
            final_dist, dist_hist, h_hist = run_single_ea(
                cx_name=op, seed=seed, pop_size=POP_SIZE, n_gens=N_GENS
            )
            results_data[op].append(final_dist)
            convergence_histories[op].append(dist_hist)
            heritability_histories[op].append(h_hist)

        mean_final = np.mean(results_data[op])
        median_final = np.median(results_data[op])
        print(f"    -> Completed {op}: Median = {median_final:.1f}, Mean = {mean_final:.1f}")

    # Compute average performance across all 30 runs for each generation
    mean_convergence = {op: np.mean(convergence_histories[op], axis=0) for op in OPERATORS}
    mean_heritability = {op: np.mean(heritability_histories[op], axis=0) for op in OPERATORS}

    # Display Statistical Summary Table
    print("\n" + "=" * 75)
    print("  STATISTICAL EVALUATION & EFFECT SIZE (30 Independent Runs)")
    print("=" * 75)

    summary_rows = []
    for op in OPERATORS:
        arr = np.array(results_data[op])
        summary_rows.append({
            "Operator": op,
            "Median": np.median(arr),
            "Mean": np.mean(arr),
            "Std": np.std(arr),
            "Min": np.min(arr),
            "Max": np.max(arr)
        })

    header = f"{'Operator':<10} | {'Median':<9} | {'Mean':<9} | {'Std':<8} | {'Min':<8} | {'Max':<8}"
    print(header)
    print("-" * len(header))
    for r in summary_rows:
        print(f"{r['Operator']:<10} | {r['Median']:<9.1f} | {r['Mean']:<9.1f} | {r['Std']:<8.1f} | {r['Min']:<8.0f} | {r['Max']:<8.0f}")

    # Mann-Whitney U Tests & Effect Sizes
    print("\n--- Hypothesis Testing (Mann-Whitney U & Vargha-Delaney A12) ---")
    er_vals = results_data["ER"]
    for comp_op in ["PMX", "OX"]:
        comp_vals = results_data[comp_op]
        stat_val, p_val = stats.mannwhitneyu(er_vals, comp_vals, alternative="two-sided")
        a12 = vargha_delaney_a12(er_vals, comp_vals)
        print(f"ER vs {comp_op:3s} | p-value: {p_val:.4e} | A12 Effect Size: {a12:.4f} "
              f"({'Statistically Significant' if p_val < 0.05 else 'Not Significant'})")

    # Export CSV File
    with open("tsp_results_summary.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Operator", "Median", "Mean", "Std", "Min", "Max"])
        for r in summary_rows:
            writer.writerow([r["Operator"], r["Median"], r["Mean"], r["Std"], r["Min"], r["Max"]])
    print("\n[✓] Results exported to 'tsp_results_summary.csv'")

In [31]:
def main():
    print("=" * 75)
    print("  TSP CROSSOVER OPERATOR COMPARISON: ulysses22 (TSPLIB)")
    print("  Target Optimal Distance: 7013")
    print("=" * 75)

    OPERATORS = ["ER", "PMX", "OX"]
    N_RUNS = 30
    N_GENS = 1000
    POP_SIZE = 100
    BASE_SEED = 2026

    results_data = {op: [] for op in OPERATORS}
    convergence_histories = {op: [] for op in OPERATORS}
    heritability_histories = {op: [] for op in OPERATORS}

    # Run 30 Seeds per Operator (90 Total Evolutionary Runs)
    for op in OPERATORS:
        print(f"\n[+] Executing {N_RUNS} independent runs for operator: {op} ...")
        for r in range(N_RUNS):
            seed = BASE_SEED + r
            final_dist, dist_hist, h_hist = run_single_ea(
                cx_name=op, seed=seed, pop_size=POP_SIZE, n_gens=N_GENS
            )
            results_data[op].append(final_dist)
            convergence_histories[op].append(dist_hist)
            heritability_histories[op].append(h_hist)

        mean_final = np.mean(results_data[op])
        median_final = np.median(results_data[op])
        print(f"    -> Completed {op}: Median = {median_final:.1f}, Mean = {mean_final:.1f}")

    # Compute average performance across all 30 runs for each generation
    mean_convergence = {op: np.mean(convergence_histories[op], axis=0) for op in OPERATORS}
    mean_heritability = {op: np.mean(heritability_histories[op], axis=0) for op in OPERATORS}

    # Display Statistical Summary Table
    print("\n" + "=" * 75)
    print("  STATISTICAL EVALUATION & EFFECT SIZE (30 Independent Runs)")
    print("=" * 75)

    summary_rows = []
    for op in OPERATORS:
        arr = np.array(results_data[op])
        summary_rows.append({
            "Operator": op,
            "Median": np.median(arr),
            "Mean": np.mean(arr),
            "Std": np.std(arr),
            "Min": np.min(arr),
            "Max": np.max(arr)
        })

    header = f"{'Operator':<10} | {'Median':<9} | {'Mean':<9} | {'Std':<8} | {'Min':<8} | {'Max':<8}"
    print(header)
    print("-" * len(header))
    for r in summary_rows:
        print(f"{r['Operator']:<10} | {r['Median']:<9.1f} | {r['Mean']:<9.1f} | {r['Std']:<8.1f} | {r['Min']:<8.0f} | {r['Max']:<8.0f}")

    # Mann-Whitney U Tests & Effect Sizes
    print("\n--- Hypothesis Testing (Mann-Whitney U & Vargha-Delaney A12) ---")
    er_vals = results_data["ER"]
    for comp_op in ["PMX", "OX"]:
        comp_vals = results_data[comp_op]
        stat_val, p_val = stats.mannwhitneyu(er_vals, comp_vals, alternative="two-sided")
        a12 = vargha_delaney_a12(er_vals, comp_vals)
        print(f"ER vs {comp_op:3s} | p-value: {p_val:.4e} | A12 Effect Size: {a12:.4f} "
              f"({'Statistically Significant' if p_val < 0.05 else 'Not Significant'})")

    # Export CSV File
    with open("tsp_results_summary.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Operator", "Median", "Mean", "Std", "Min", "Max"])
        for r in summary_rows:
            writer.writerow([r["Operator"], r["Median"], r["Mean"], r["Std"], r["Min"], r["Max"]])
    print("\n[✓] Results exported to 'tsp_results_summary.csv'")

    # STAGE 8: PLOT GENERATION

    gens_axis = np.arange(1, N_GENS + 1)
    colors = {"ER": "#1f77b4", "PMX": "#ff7f0e", "OX": "#2ca02c"}

    # Figure 1: Tour Distance Convergence Curves
    plt.figure(figsize=(9, 5.5), dpi=300)
    for op in OPERATORS:
        plt.plot(gens_axis, mean_convergence[op], label=f"{op} Crossover", color=colors[op], linewidth=2.0)
    plt.axhline(y=7013, color='black', linestyle='--', linewidth=1.5, label="TSPLIB Optimal (7013)")
    plt.xlabel("Generation", fontsize=12)
    plt.ylabel("Mean Best Tour Distance", fontsize=12)
    plt.title("ulysses22: Tour Distance Convergence Comparison", fontsize=14, fontweight='bold')
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig("tsp_convergence.png")
    plt.close()
    print("[✓] Plot saved: 'tsp_convergence.png'")

    # Figure 2: Mean Edge Heritability vs Generation Curve
    plt.figure(figsize=(9, 5.5), dpi=300)
    for op in OPERATORS:
        plt.plot(gens_axis, mean_heritability[op], label=f"{op} Crossover", color=colors[op], linewidth=2.0)
    plt.xlabel("Generation", fontsize=12)
    plt.ylabel(r"Mean Edge Heritability $H(C \mid P_1, P_2)$", fontsize=12)
    plt.title("ulysses22: Edge Heritability Across Generations", fontsize=14, fontweight='bold')
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig("tsp_heritability.png")
    plt.close()
    print("[✓] Plot saved: 'tsp_heritability.png'")

    # Figure 3: Boxplot of Final Tour Length Distributions
    plt.figure(figsize=(7, 5), dpi=300)
    data_to_plot = [results_data[op] for op in OPERATORS]
    bp = plt.boxplot(data_to_plot, patch_artist=True, tick_labels=OPERATORS)
    for patch, op in zip(bp['boxes'], OPERATORS):
        patch.set_facecolor(colors[op])
        patch.set_alpha(0.7)
    plt.axhline(y=7013, color='black', linestyle='--', linewidth=1.5, label="Optimal (7013)")
    plt.ylabel("Final Best Tour Distance", fontsize=12)
    plt.title("ulysses22: Final Distance Distribution (30 Runs)", fontsize=13, fontweight='bold')
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig("tsp_boxplot.png")
    plt.close()
    print("[✓] Plot saved: 'tsp_boxplot.png'")

    print("\n[+] All tasks complete! You are ready to embed the results into your PDF report.")

if __name__ == "__main__":
    main()

  TSP CROSSOVER OPERATOR COMPARISON: ulysses22 (TSPLIB)
  Target Optimal Distance: 7013

[+] Executing 30 independent runs for operator: ER ...
    -> Completed ER: Median = 4521.0, Mean = 4563.0

[+] Executing 30 independent runs for operator: PMX ...
    -> Completed PMX: Median = 4544.0, Mean = 4589.7

[+] Executing 30 independent runs for operator: OX ...
    -> Completed OX: Median = 4498.0, Mean = 4510.0

  STATISTICAL EVALUATION & EFFECT SIZE (30 Independent Runs)
Operator   | Median    | Mean      | Std      | Min      | Max     
-------------------------------------------------------------------
ER         | 4521.0    | 4563.0    | 85.6     | 4498     | 4761    
PMX        | 4544.0    | 4589.7    | 101.6    | 4498     | 4761    
OX         | 4498.0    | 4510.0    | 24.2     | 4498     | 4561    

--- Hypothesis Testing (Mann-Whitney U & Vargha-Delaney A12) ---
ER vs PMX | p-value: 4.0346e-01 | A12 Effect Size: 0.4400 (Not Significant)
ER vs OX  | p-value: 6.0271e-03 | A12 Effe